In [1]:
# CREATE TABLE IF NOT EXISTS ICEBERG_DB.TBL_1(
#     ID INT,
#     NAME STRING,
#     AGE INT,
#     EMAIL STRING
# )
# LOCATION 's3://shivchoudhury-datasets/iceberg-tbl/TBL_1/'
# TBLPROPERTIES ('table_type'='ICEBERG');

# SELECT * FROM ICEBERG_DB.TBL_1;

# CREATE EXTERNAL TABLE IF NOT EXISTS my_glue_db.csv_tbl(
#     ID INT,
#     NAME STRING,
#     AGE INT,
#     EMAIL STRING
# )
# ROW FORMAT DELIMITED FIELDS TERMINATED BY ','
# STORED AS TEXTFILE
# LOCATION 's3://shivchoudhury-datasets/iceberg-tbl/csv_tbl/'
# TBLPROPERTIES ("skip.header.line.count"="1");

# SELECT * FROM my_glue_db.csv_tbl;

# INSERT INTO iceberg_db.tbl_1
# SELECT * FROM my_glue_db.csv_tbl;

# pyspark --packages org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.3.1

In [29]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit

In [30]:
spark = (SparkSession.builder
            .appName("DeltaTableFormats LakehouseApp")
            .master("local[*]")
            .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0," \
                                            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0," \
                                            "org.apache.hudi:hudi-spark3.5-bundle_2.12:0.15.0," \
                                            "org.apache.hadoop.fs.s3a.S3AFileSystem," \
                                            "org.apache.hadoop:hadoop-aws:3.3.4," \
                                            "com.amazonaws:aws-java-sdk-bundle:1.12.262," \
                                            "software.amazon.awssdk:glue:2.25.26," \
                                            "software.amazon.awssdk:sts:2.25.26," \
                                            "software.amazon.awssdk:s3:2.25.26," \
                                            "software.amazon.awssdk:dynamodb:2.25.26," \
                                            "org.apache.iceberg:iceberg-aws-bundle:1.5.0," \
                                            "software.amazon.awssdk:url-connection-client:2.25.26")
            .config("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
            .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension,org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
            .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
            .config("spark.sql.catalog.glue_catalog",  "org.apache.iceberg.spark.SparkCatalog")
            .config("spark.sql.catalog.glue_catalog.warehouse", "s3://shivchoudhury-datasets/warehouse/")
            .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
            .config("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
            .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")
            .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
            .config("spark.hadoop.fs.s3a.endpoint", "s3.ap-south-1.amazonaws.com")
            .config("spark.sql.catalog.glue_catalog.client.region", "ap-south-1")
            .config("spark.sql.defaultCatalog", "glue_catalog")
            .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
            .config("spark.hadoop.fs.s3a.path.style.access", "true")
            .config("hive.metastore.client.factory.class", "com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory") \
            .enableHiveSupport()
            .getOrCreate()
        );

In [31]:
spark

In [32]:
source_path = 's3://shivchoudhury-datasets/MergeSchema/paquet-customers/'

In [33]:
df = spark.read.parquet(source_path)

In [34]:
df.show()

+-----------+----------+----------+---------+---------+-----------+----------+-------------+--------------------+
|CUSTOMER_ID|SALUTATION|FIRST_NAME|LAST_NAME|BIRTH_DAY|BIRTH_MONTH|BIRTH_YEAR|BIRTH_COUNTRY|       EMAIL_ADDRESS|
+-----------+----------+----------+---------+---------+-----------+----------+-------------+--------------------+
|   4YZCNG66|     Prof.|      Alex|    Brown|       15|          6|      1960|    Australia| alex.brown@test.org|
|   4EYG05GW|     Prof.|      Alex| Martinez|        7|          4|      2003|       Brazil|alex.martinez@com...|
|   PI7I3BQ1|      Mrs.|      John|   Miller|       24|          3|      1992|      Germany|john.miller@compa...|
|   YX2Y51U5|       Ms.|    Morgan|    Davis|       25|          6|      1966|        China|morgan.davis@comp...|
|   IHAQFM5I|     Prof.|     Chris|Rodriguez|        6|         10|      1983|    Australia|chris.rodriguez@t...|
|   0N18MDGN|     Prof.|      Alex| Martinez|       14|          7|      1982|        In

In [35]:
spark.sql("CREATE DATABASE IF NOT EXISTS spark_tuts")

26/04/28 19:04:00 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/04/28 19:04:00 WARN HiveConf: HiveConf of name hive.metastore.client.factory.class does not exist
26/04/28 19:04:00 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/04/28 19:04:11 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/04/28 19:04:11 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore UNKNOWN@172.21.0.2
26/04/28 19:04:12 WARN ObjectStore: Failed to get database spark_tuts, returning NoSuchObjectException
26/04/28 19:04:12 WARN ObjectStore: Failed to get database spark_tuts, returning NoSuchObjectException
26/04/28 19:04:12 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException
26/04/28 19:04:12 WARN ObjectStore: Failed to get database spark_tuts, retur

DataFrame[]

In [51]:
spark.sql("SELECT current_catalog()").show()

+-----------------+
|current_catalog()|
+-----------------+
|     glue_catalog|
+-----------------+



In [52]:
spark.sql("USE glue_catalog")

DataFrame[]

In [66]:
spark.sql("SHOW DATABASES").show()

+----------+
| namespace|
+----------+
|   default|
|      hive|
|   hudi_db|
|iceberg_db|
|   kinesis|
|my_glue_db|
|     sales|
|spark_tuts|
+----------+



In [61]:
spark.sql("CREATE DATABASE IF NOT EXISTS glue_catalog.spark_tuts")

DataFrame[]

In [54]:
spark.sql("DROP TABLE IF EXISTS glue_catalog.spark_tuts.customer")

DataFrame[]

In [64]:
spark.sql("SHOW databases").show()

+----------+
| namespace|
+----------+
|   default|
|      hive|
|   hudi_db|
|iceberg_db|
|   kinesis|
|my_glue_db|
|     sales|
|spark_tuts|
+----------+



In [65]:
df.write.format("iceberg").mode("append").saveAsTable("glue_catalog.spark_tuts.customer")

26/04/28 19:11:56 WARN ProfileFileReader: Ignoring earlier-seen '[default]', because '[profile default]' was found on line 5
26/04/28 19:11:58 WARN ProfileFileReader: Ignoring earlier-seen '[default]', because '[profile default]' was found on line 5
26/04/28 19:11:58 WARN ProfileFileReader: Ignoring earlier-seen '[default]', because '[profile default]' was found on line 5
                                                                                

In [67]:
spark.stop()